In [1]:
import pypdf

# Path to test document inside documents/
pdf_path = "documents/1706.03762v7.pdf"

reader = pypdf.PdfReader(pdf_path)
print(f"Total pages: {len(reader.pages)}")

# Sanity check: preview first page content
first_page_text = reader.pages[0].extract_text()
print(first_page_text[:500])

Total pages: 15
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz K


In [2]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter

documents_folder = "documents"

# Collect every PDF in the folder automatically
pdf_files = [f for f in os.listdir(documents_folder) if f.endswith(".pdf")]
print(f"Found {len(pdf_files)} PDF files: {pdf_files}")

# Extract full text from each PDF
all_documents = []
for filename in pdf_files:
    filepath = os.path.join(documents_folder, filename)
    reader = pypdf.PdfReader(filepath)

    full_text = ""
    for page in reader.pages:
        full_text += page.extract_text() + "\n"

    all_documents.append({"filename": filename, "text": full_text})
    print(f"{filename}: {len(full_text)} characters extracted")

Found 2 PDF files: ['1706.03762v7.pdf', '_OceanofPDF.com_Raldoron_Revenant_-_Guy_Haley.pdf']
1706.03762v7.pdf: 39511 characters extracted
_OceanofPDF.com_Raldoron_Revenant_-_Guy_Haley.pdf: 324245 characters extracted


In [7]:
# Chunk per-page (not the whole document) so each chunk knows its source page.
# This slightly reduces cross-page context but makes citations exact.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

all_chunks = []
for filename in pdf_files:
    filepath = os.path.join(documents_folder, filename)
    reader = pypdf.PdfReader(filepath)

    for page_number, page in enumerate(reader.pages, start=1):
        page_text = page.extract_text()
        if not page_text.strip():
            continue  # skip empty/scanned pages with no extractable text

        page_chunks = text_splitter.split_text(page_text)
        for chunk in page_chunks:
            all_chunks.append({
                "filename": filename,
                "page": page_number,
                "text": chunk,
            })

print(f"Total chunks created: {len(all_chunks)}")
print(f"\n--- Sample chunk ---")
print(f"Source: {all_chunks[5]['filename']}, page {all_chunks[5]['page']}")
print(all_chunks[5]["text"][:300])

Total chunks created: 495

--- Sample chunk ---
Source: 1706.03762v7.pdf, page 2
sequence lengths, as memory constraints limit batching across examples. Recent work has achieved
significant improvements in computational efficiency through factorization tricks [21] and conditional
computation [32], while also improving model performance in case of the latter. The fundamental
cons


In [8]:
# Reset the collection: previous version lacked page-level metadata.
# Safe to run even if the collection doesn't exist yet.
chroma_client_temp = chromadb.PersistentClient(path="./chroma_db")
try:
    chroma_client_temp.delete_collection(name="doculocal_rag")
    print("Old collection deleted.")
except Exception:
    print("No existing collection to delete — starting fresh.")

Old collection deleted.


In [9]:
import chromadb
from chromadb.utils import embedding_functions

# Ollama-backed embedding function: converts text into vectors locally
ollama_ef = embedding_functions.OllamaEmbeddingFunction(
    url="http://localhost:11434/api/embeddings",
    model_name="nomic-embed-text",
)

# Persistent client: data survives between notebook restarts
chroma_client = chromadb.PersistentClient(path="./chroma_db")

collection = chroma_client.get_or_create_collection(
    name="doculocal_rag",
    embedding_function=ollama_ef,
)

print(f"Collection ready. Current items: {collection.count()}")

Collection ready. Current items: 0


In [12]:
# Send data in batches to avoid overwhelming the embedding server with one huge request
BATCH_SIZE = 50

ids = [f"chunk_{i}" for i in range(len(all_chunks))]
texts = [chunk["text"] for chunk in all_chunks]
metadatas = [
    {"source": chunk["filename"], "page": chunk["page"]}
    for chunk in all_chunks
]

for start in range(0, len(all_chunks), BATCH_SIZE):
    end = start + BATCH_SIZE
    collection.add(
        ids=ids[start:end],
        documents=texts[start:end],
        metadatas=metadatas[start:end],
    )
    print(f"Added batch {start}-{end} ({collection.count()} total so far)")

print(f"\nCollection now contains: {collection.count()} items")

Added batch 0-50 (495 total so far)
Added batch 50-100 (495 total so far)
Added batch 100-150 (495 total so far)
Added batch 150-200 (495 total so far)
Added batch 200-250 (495 total so far)
Added batch 250-300 (495 total so far)
Added batch 300-350 (495 total so far)
Added batch 350-400 (495 total so far)
Added batch 400-450 (495 total so far)
Added batch 450-500 (495 total so far)

Collection now contains: 495 items


In [13]:
# Verify metadata structure directly from the stored collection
sample = collection.get(ids=["chunk_10"], include=["metadatas", "documents"])
print("Metadata:", sample["metadatas"][0])
print("\nText preview:", sample["documents"][0][:200])

Metadata: {'page': 3, 'source': '1706.03762v7.pdf'}

Text preview: layers, produce outputs of dimensiondmodel = 512.
Decoder: The decoder is also composed of a stack ofN = 6 identical layers. In addition to the two
sub-layers in each encoder layer, the decoder insert
